# Lecture 3 — Restore a noisy image with an Ising prior

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opherdonchin/AdvancedStatisticsCourse_36725561/blob/main/notebooks/lecture3_restoration.ipynb)


A camera records a black-and-white image, but every pixel reading is corrupted by Gaussian noise. Reading each pixel on its own — the lecture's $\operatorname{sign}(y_i)$ — ignores the fact that neighbouring pixels in real images tend to agree. **Does coupling neighbouring pixels through the lecture's Ising prior reduce restoration error, and what exactly does the resulting algorithm compute?**

| Section | What we do | Contribution to the question |
|---|---|---|
| 1. Add the data to the conditional for one pixel | See how each noisy reading changes the lecture's conditional for one pixel; weigh the reading against the neighbours | One number per pixel, computed from its neighbours and its reading, that every later step uses |
| 2. Restore the image one pixel at a time (ICM) | Set each pixel to its most probable value given its neighbours, repeatedly | A restoration method and its guarantee |
| 3. Compare with reading each pixel on its own | Count errors against the clean image; vary the coupling $J$; choose $J$ on a separate calibration image | Whether coupling helps, and how to report it honestly |
| 4. What was computed, and what the prior assumes | Examine a run that never moves, and a feature the prior erases | What the answer means and where the model fails |

The image is course-constructed. We keep the clean image only to **score** restorations: it never enters the model, the starting image, the update or the choice of $J$, because in a real restoration it would be unknown.

### *Code to run*

#### Set up the image and its noisy observation

The clean image `clean` has pixels $z_i\in\{-1,+1\}$ (dark $-1$, light $+1$): a disk, a square ring, a two-pixel-wide line and a one-pixel-wide line. The observation is $y_i=z_i+\varepsilon_i$ with independent $\varepsilon_i\sim\mathcal N(0,\sigma^2)$ and $\sigma^2=0.8$, as on the lecture slides. `baseline` is the pixel-by-pixel restoration $\operatorname{sign}(y_i)$, which the lecture showed is the image that maximizes the likelihood $p(\mathbf y\mid\mathbf z)$ (S5.03); bold $\mathbf z,\mathbf y$ denote whole images, as on the slides. `np.mgrid` returns the row and column index of every pixel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rows, cols = np.mgrid[0:48, 0:64]
clean = -np.ones((48, 64), dtype=int)
clean[(rows - 24) ** 2 + (cols - 16) ** 2 <= 11.5 ** 2] = 1               # disk
ring = (rows >= 8) & (rows <= 40) & (cols >= 34) & (cols <= 58)
hole = (rows >= 16) & (rows <= 32) & (cols >= 42) & (cols <= 50)
clean[ring & ~hole] = 1                                                   # square ring
clean[2:4, 4:30] = 1                                                      # two-pixel line
clean[44, 4:60] = 1                                                       # one-pixel line

variance = 0.8                                                            # sigma^2
rng = np.random.default_rng(1)
y = clean + rng.normal(0.0, np.sqrt(variance), clean.shape)
baseline = np.where(y > 0, 1, -1)

print(f"{clean.size} pixels; largest reading y = {y.max():.2f}")

### *Code to run*

#### Display images

`show` draws a $\pm1$ image (or the readings $y$, on a wider grey scale) on a given axis. `pixel_errors` counts pixels where a restoration differs from the clean image; it is used only for scoring.

In [ ]:
def show(ax, image, title, readings=False):
    limit = 2 if readings else 1
    ax.imshow(image, cmap="gray", vmin=-limit, vmax=limit, interpolation="nearest")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])


def pixel_errors(z):
    return int((z != clean).sum())


fig, axes = plt.subplots(1, 3, figsize=(10, 2.8))
show(axes[0], clean, "clean z (scoring only)")
show(axes[1], y, "observed y", readings=True)
show(axes[2], baseline, f"sign(y): {pixel_errors(baseline)} errors")
plt.show()

## 1. Add the data to the lecture's conditional for one pixel

### *Explanation to read*

#### Recall the model

We want the clean pixel values $z_i\in\{-1,+1\}$ and we see one noisy reading $y_i$ per pixel. Bold $\mathbf z$ and $\mathbf y$ denote the whole images. The lecture built the model from two kinds of factors.

**Pair weights: the prior (S5.05–S5.06).** For every pair of neighbouring pixels $i\sim j$ (each pair counted once),
$$
\psi_{ij}(z_i,z_j)=e^{Jz_iz_j}. \tag{1}
$$
It is $e^{J}$ when the two pixels agree and $e^{-J}$ when they disagree, so with $J>0$ images whose neighbours agree get more weight.

**Data weights: the likelihood (S5.03).** The data weight chosen in the lecture is the Gaussian density of the reading given the pixel's value:
$$
\varphi_i(z_i)=\mathcal N(y_i\mid z_i,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left[-\frac{(y_i-z_i)^2}{2\sigma^2}\right]. \tag{2}
$$
It is larger for whichever value of $z_i$ lies closer to the reading $y_i$.

**Posterior (S5.07).** Multiplying all the factors gives the posterior over clean images, up to a constant:
$$
p(\mathbf z\mid\mathbf y)\;\propto\;\prod_{i\sim j}\psi_{ij}(z_i,z_j)\;\prod_i\varphi_i(z_i). \tag{3}
$$

### *Explanation to read*

#### What the lecture computed for one pixel

For the prior alone, the lecture (S6.02) asked about a single pixel $i$ with all the other pixels $\mathbf z_{-i}$ held fixed. The conditional probability divides the probability of one image by the total over the two images that differ only at pixel $i$:
$$
P(z_i=+1\mid\mathbf z_{-i})=\frac{p(+1,\mathbf z_{-i})}{\sum_{s=\pm1}p(s,\mathbf z_{-i})}.
$$
Each of these probabilities is a product of three parts: $Z^{-1}$, the pair weights linking $i$ to its neighbours $\mathrm{ne}(i)$, and $R$, the product of all the other pair weights (those of pairs that do not include $i$). Neither $Z$ nor $R$ depends on $z_i$. So $Z^{-1}R$ is the same in the numerator and the denominator, and it cancels. The neighbours' pair weights multiply to $e^{z_ih_i}$, and the result was
$$
P(z_i=+1\mid \mathbf z_{-i})=\frac{1}{1+e^{-2h_i}},\qquad h_i=J\sum_{j\in\mathrm{ne}(i)}z_j . \tag{4}
$$
You can read $h_i$ as the neighbours' vote: each light neighbour adds $J$, each dark one subtracts $J$. The distribution of one pixel given all the others is called its **site conditional** (a *site* is one pixel position); (4) is the site conditional under the prior alone.

### *Explanation to read*

#### Adding the data weight

With data, the pair weights linking $i$ to its neighbours are no longer the only factors that involve $z_i$: the posterior (3) has exactly one other, the pixel's data weight (2). Because $z_i$ is $\pm1$, we can write that factor in the same exponential form as the pair weights. Expanding the square, $(y_i-z_i)^2=y_i^2-2y_iz_i+1$ since $z_i^2=1$, so
$$
\varphi_i(z_i)=c_i\,e^{y_iz_i/\sigma^2},\qquad c_i=\frac{e^{-(y_i^2+1)/(2\sigma^2)}}{\sqrt{2\pi\sigma^2}}\ \text{(the same for both values of } z_i\text{)}. \tag{5}
$$
Repeating the lecture's comparison with this extra factor, $c_i$ cancels too, and the site conditional has the same form as (4), with one more term in the field than in (4):
$$
P(z_i=+1\mid \mathbf z_{-i},\mathbf y)=\frac{1}{1+e^{-2h_i}},\qquad h_i=J\sum_{j\in\mathrm{ne}(i)}z_j+\frac{y_i}{\sigma^2}. \tag{6}
$$
We call $h_i$ the **posterior field** of pixel $i$; from here on, $h_i$ always means (6). The reading casts one more vote: positive readings vote light, negative readings vote dark, and noisier measurements (larger $\sigma^2$) vote more weakly.

### *Questions to answer*

#### 1. Weigh the neighbours against the reading — written answer

Use the posterior field (6) with the lecture's values $J=0.5$ and $\sigma^2=0.8$. Pixel $i$ has all four neighbours light ($+1$), but its reading is $y_i=-1$, exactly what a dark pixel would give with no noise. What is $P(z_i=+1\mid\mathbf z_{-i},\mathbf y)$, and which does the model trust more here, the neighbours or the reading? (You may use Python as a calculator.)

- answer here

### *Explanation to read*

#### One number scores a whole image

Taking the logarithm of the posterior (3) and using (5) for every data weight gives one number for any candidate image, its **image score**:
$$
\log p(\mathbf z\mid\mathbf y)=F(\mathbf z)+\text{const},\qquad
F(\mathbf z)=J\sum_{i\sim j}z_iz_j+\frac{1}{\sigma^2}\sum_i y_iz_i . \tag{7}
$$
The constant does not depend on $\mathbf z$, so $F$ ranks images exactly as the posterior does: the most probable image (the MAP) is the one with the largest $F$.

Only a few terms of $F$ involve a given pixel $i$: its pairs with its neighbours and its own reading. Together they are $Jz_i\sum_{j\in\mathrm{ne}(i)}z_j+y_iz_i/\sigma^2=z_ih_i$. So flipping pixel $i$ from $-1$ to $+1$, with every other pixel fixed, changes the image score by
$$
F(\mathbf z\text{ with }z_i=+1)-F(\mathbf z\text{ with }z_i=-1)=2h_i . \tag{8}
$$
The check after Question 2 uses (8) to test your code.

### *Code to run*

#### The image score and the neighbour sum

`objective` computes the image score $F(\mathbf z)$ of (7); `z[1:, :] * z[:-1, :]` multiplies every vertical pair and `z[:, 1:] * z[:, :-1]` every horizontal pair. `neighbour_sum(z, r, c)` returns $\sum_{j\in\mathrm{ne}(i)}z_j$ for the pixel $i$ at row `r`, column `c`; pixels on the border have only two or three neighbours.

In [ ]:
def objective(z, y, J, variance):
    pairs = (z[1:, :] * z[:-1, :]).sum() + (z[:, 1:] * z[:, :-1]).sum()
    return J * pairs + (y * z).sum() / variance


def neighbour_sum(z, r, c):
    n_rows, n_cols = z.shape
    total = 0
    if r > 0:
        total += z[r - 1, c]
    if r < n_rows - 1:
        total += z[r + 1, c]
    if c > 0:
        total += z[r, c - 1]
    if c < n_cols - 1:
        total += z[r, c + 1]
    return total


print("F of the sign(y) image at J = 0.5:", round(objective(baseline, y, 0.5, variance), 1))

### *Questions to answer*

#### 2. Implement the posterior field — code

Write `local_field(z, y, r, c, J, variance)` returning the posterior field $h_i$ of equation (6) for the pixel $i$ at row `r`, column `c` of the current image `z`. Use `neighbour_sum`.

In [ ]:
# answer here

### *Code to run*

#### Check your field against the image score

At 200 random pixels of a random image, this checks identity (8): it compares $F(\mathbf z\text{ with }z_i=+1)-F(\mathbf z\text{ with }z_i=-1)$ with $2\times$ your `local_field`. The largest discrepancy should be at rounding level (about $10^{-13}$ or smaller).

In [ ]:
check_rng = np.random.default_rng(0)
z_random = check_rng.choice([-1, 1], size=clean.shape)
largest = 0.0
for _ in range(200):
    r, c = check_rng.integers(48), check_rng.integers(64)
    plus, minus = z_random.copy(), z_random.copy()
    plus[r, c], minus[r, c] = 1, -1
    gap = objective(plus, y, 0.5, variance) - objective(minus, y, 0.5, variance)
    largest = max(largest, abs(gap - 2 * local_field(z_random, y, r, c, 0.5, variance)))
assert largest < 1e-9
print(f"Largest discrepancy over 200 pixels: {largest:.1e}")

### *Explanation to read*

#### Section summary

Each noisy reading $y_i$ adds one factor to the posterior, its data weight $e^{y_iz_i/\sigma^2}$ (5), and that factor involves only $z_i$, the pixel the reading measures. Including it in the prior's site conditional (4) gives the posterior site conditional (6), whose field $h_i$ is the neighbours' vote plus the pixel's own reading. Four agreeing neighbours can outvote a single dark reading. Computing $h_i$ reads only the pixel's four neighbours and its own reading, so it takes the same few arithmetic operations however large the image is; and $2h_i$ is exactly the change in the image score $F$ when that pixel flips.

## 2. Restore the image one pixel at a time (ICM)

### *Explanation to read*

#### Iterated conditional modes

For a row of three pixels, the lecture listed all eight images and computed their posterior probabilities directly (S5.09). Our image has 3,072 pixels and so $2^{3072}$ possible images: we cannot list them, and restoring the scan means choosing one image from the posterior some other way. A simple local method is **iterated conditional modes (ICM)**, due to Besag (1986): visit the pixels in a fixed order (here row by row) and set each one to the most probable value of its posterior site conditional given the current values of its neighbours. Since $P(z_i=+1\mid \mathbf z_{-i},\mathbf y)>1/2$ exactly when $h_i>0$, the update is
$$
z_i\leftarrow\begin{cases}+1 & h_i>0\\ -1 & h_i<0\\ z_i\ (\text{unchanged}) & h_i=0.\end{cases} \tag{9}
$$
One pass over all pixels is a **sweep**. Each update uses neighbours already updated in the same sweep. ICM stops when a whole sweep changes no pixel.

### *Questions to answer*

#### 3. Implement the site update — code

Write `icm_update(current, h)` returning the new value of a pixel whose current value is `current` and whose posterior field is `h`, following the update rule (9).

In [ ]:
# answer here

### *Code to run*

#### The sweep driver

`icm` copies the starting image, sweeps row by row calling your `local_field` and `icm_update`, records the image score $F$ after each sweep in `history`, and stops after a sweep with no change.

In [ ]:
def icm(z_start, y, J, variance, max_sweeps=100):
    z = z_start.copy()
    history = [objective(z, y, J, variance)]
    for _ in range(max_sweeps):
        changed = 0
        for r in range(z.shape[0]):
            for c in range(z.shape[1]):
                new = icm_update(z[r, c], local_field(z, y, r, c, J, variance))
                changed += new != z[r, c]
                z[r, c] = new
        history.append(objective(z, y, J, variance))
        if changed == 0:
            return z, history
    raise RuntimeError("ICM did not stop")

### *Code to run*

#### Restore the image with $J=0.5$

$J=0.5$ is the lecture's coupling, fixed before looking at any result. The run starts from `baseline`. The right panel shows $F$ after each sweep.

In [ ]:
restored, history = icm(baseline, y, 0.5, variance)

fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), gridspec_kw={"width_ratios": [1, 1, 1.1]})
show(axes[0], baseline, "start: sign(y)")
show(axes[1], restored, f"ICM, J = 0.5 ({len(history) - 1} sweeps)")
axes[2].plot(history, marker="o")
axes[2].set(xlabel="sweep", ylabel="F(z)", title="image score after each sweep")
plt.show()
print("F after each sweep:", np.round(history, 1))

### *Questions to answer*

#### 4. Explain why the image score never decreases — written answer

Using identity (8) (flipping pixel $i$ from $-1$ to $+1$ changes $F$ by $2h_i$), explain why a single ICM update can never decrease $F$. Why must ICM therefore stop after finitely many sweeps?

- answer here

### *Explanation to read*

#### Section summary

ICM restores the image one pixel at a time. It visits the pixels in turn and sets each one to its more probable value under the posterior site conditional (6), given the current values of its neighbours: by rule (9), $\operatorname{sign}(h_i)$. Each update computes $h_i$ from four neighbours and one reading, so a sweep takes time proportional to the number of pixels. No update ever lowers the image score $F$, and a pixel changes only when $F$ strictly rises, so the algorithm stops — here after five sweeps. This holds in any visiting order, because it holds for every single update, but a different order can stop at a different image (63 to 77 errors over 20 random orders at $J=0.5$, computed for this notebook). We fix row-by-row order so that everyone's results can be checked against the same numbers.

## 3. Compare with reading each pixel on its own

### *Questions to answer*

#### 5. Count the errors — code

Run `icm` from `baseline` with $J=0$, storing the result as `restored_J0`. Then print the pixel errors (`pixel_errors`) of `baseline`, `restored_J0` and `restored` (the $J=0.5$ result).

In [ ]:
# answer here

### *Explanation to read*

#### Why $J=0$ reproduces the baseline

You should find 397 errors for both $\operatorname{sign}(y)$ and $J=0$, and 72 for $J=0.5$. With $J=0$ the field is $h_i=y_i/\sigma^2$, so the update is $\operatorname{sign}(y_i)$: the pixel-by-pixel restoration is the no-coupling case of the same model, and the likelihood maximizer of S5.03. Coupling with $J=0.5$ removes about 80% of its errors.

How much does that depend on the choice $J=0.5$, and on starting from $\operatorname{sign}(y)$? The next cell restores the image for five values of $J$ from two starting images: $\operatorname{sign}(y)$ and a blank page (all $-1$). It is a **sensitivity check** of the method, not yet a way of choosing $J$.

### *Code to run*

#### Vary the coupling and the starting image

Each panel shows the ICM result, its pixel errors and its final $F$; the printed table also counts errors on the 56-pixel one-pixel line. This takes a few seconds.

In [ ]:
J_values = [0.0, 0.25, 0.5, 1.0, 2.0]
starts = {"sign(y)": baseline, "blank": -np.ones_like(clean)}
fig, axes = plt.subplots(2, len(J_values), figsize=(13, 5.6))
for col, J in enumerate(J_values):
    for row, (name, start) in enumerate(starts.items()):
        z, h = icm(start, y, J, variance)
        show(axes[row, col], z, f"J = {J}, start {name}\n{pixel_errors(z)} errors, F = {h[-1]:.0f}")
        print(f"J = {J:4}, start {name:7}: {pixel_errors(z):5} errors, F = {h[-1]:8.1f}, "
              f"{len(h) - 1:2} sweeps, one-pixel line errors {int((z[44, 4:60] != 1).sum()):2}/56")
plt.show()

### *Questions to answer*

#### 6. Read the sensitivity check — written answer

Starting from $\operatorname{sign}(y)$, how do the pixel errors change as $J$ increases from 0 to 2, and which parts of the image are responsible at small and at large $J$?

- answer here

### *Explanation to read*

#### Choose the coupling honestly

The table suggests reporting "$J=0.5$: 72 errors", but that would overstate how well the method restores a new image. We picked $J=0.5$ by scoring five tries with the clean image, the answer key that a real restoration does not have, and then reported the error on that same image.

When some images with known truth are available, one practical way to choose $J$ is to try values on these **calibration** images and apply the best one to new images. Below is a second course-constructed scene of the same kind (rectangles, a diamond, thin lines) with its own clean image `clean_cal`, its own noise draw `y_cal` at the same $\sigma^2$, and its pixel-by-pixel restoration `baseline_cal`. The test image from the start of the notebook plays the role of the new image: its clean version may only score the final choice.

### *Code to run*

#### The calibration scene

Same construction as the test scene, with different shapes and a different noise draw.

In [ ]:
clean_cal = -np.ones((48, 64), dtype=int)
clean_cal[6:22, 6:26] = 1                                                 # rectangle
clean_cal[28:42, 10:20] = 1                                               # rectangle
clean_cal[np.abs(rows - 24) + np.abs(cols - 46) <= 12] = 1                # diamond
clean_cal[30:46, 30] = 1                                                  # one-pixel vertical line
clean_cal[4, 34:62] = 1                                                   # one-pixel horizontal line
clean_cal[42:44, 36:62] = 1                                               # two-pixel line
y_cal = clean_cal + np.random.default_rng(2).normal(0.0, np.sqrt(variance), clean_cal.shape)
baseline_cal = np.where(y_cal > 0, 1, -1)

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))
show(axes[0], clean_cal, "calibration: clean")
show(axes[1], y_cal, "calibration: observed", readings=True)
plt.show()

### *Questions to answer*

#### 7. Choose $J$ on the calibration scene and report on the test image — code

For each $J$ in `J_values`, run `icm` on the calibration readings `y_cal` starting from `baseline_cal`, and count its errors against `clean_cal` (for example `int((z != clean_cal).sum())`). Store the counts in a list `cal_errors`. Choose `J_chosen` as the value with the fewest calibration errors (`np.argmin` gives the index of the smallest entry). Then restore the test image `y` from `baseline` with `J_chosen` and print its `pixel_errors`.

In [ ]:
# answer here

### *Explanation to read*

#### Section summary

Coupling sharply reduces restoration errors, and $J=0$ recovers the pixel-by-pixel restoration exactly. Too little coupling leaves speckle; too much erases thin features. Choosing $J$ on the calibration scene gave $J=1$ and **88 errors on the test image**. Unlike "72 errors at $J=0.5$", which was chosen and scored with the same answer key, 88 is honest: the test image's truth played no part in the choice. It is not precise, though. Over 20 fresh noise draws of the test scene (computed for this notebook, not shown), $J=1$ averages about 69 errors and $J=0.5$ about 73, each varying by about $\pm12$, so the gap between 72 and 88 is mostly this draw's noise.

## 4. What ICM computed, and what the prior assumes

### *Explanation to read*

#### A run that never moves

The lecture (S6.04) separated what needs only a pixel's four neighbours, its site conditional, from what needs a sum over every image: the normalizer $Z$ and each pixel's posterior marginal probability. ICM uses only site conditionals. When it stops, every pixel already has its conditional's more probable value given its neighbours; such an image is a **fixed point** of ICM. Equivalently, no single-pixel change raises the image score: the image is a **coordinatewise optimum** of $F$.

Look at the bottom row of the sensitivity check, started from the blank page. At $J=0.5$ ICM recovers the shapes (84 errors). At $J=2$ it stops after one sweep without changing a single pixel, although the readings plainly show a disk and a ring.

### *Questions to answer*

#### 8. Why the blank page never moves at $J=2$ — written answer

Starting from the blank page, why does ICM reach a good restoration at $J=0.5$ but leave the page blank at $J=2$? Use the posterior field (6) and rule (9).

- answer here

### *Questions to answer*

#### 9. Why the thin line erodes at $J=2$ — written answer

Starting from $\operatorname{sign}(y)$, ICM gets 8 of the one-pixel line's 56 pixels wrong at $J=0.5$ but 45 at $J=2$. Why does stronger coupling erode the line? What does this say about the kind of image this prior describes?

- answer here

### *Explanation to read*

#### Answer to the opening question

**Coupling helps, within limits.** ICM reduced the errors of pixel-by-pixel restoration from 397 of 3,072 pixels to 88, with $J=1$ chosen honestly on a calibration scene (give or take about 12 from one noise draw to another). Stronger coupling erases the one-pixel line, because the prior rewards agreement with all four neighbours equally.

**What was computed.** ICM returns a coordinatewise optimum of the image score: no single-pixel change improves it, but it need not be the most probable image (the blank start stopped at a worse one). It gives no posterior probability for any pixel: the site conditional $1/(1+e^{-2h_i})$ conditions on the neighbours' current values, whereas $P(z_i=+1\mid\mathbf y)$ is a posterior marginal that sums over all the other pixels (S6.04). Finding the most probable image with certainty, or these marginals, needs a method that works on the whole image at once.

**Sources.** Murphy, *Probabilistic Machine Learning: Advanced Topics*, §4.3.2.1 (Ising model) and §4.3.8 (image restoration, Eq. 4.117, whose sum over sites should read as a product); §4.3.2.4 names ICM but writes a synchronous update, whereas this notebook uses Besag's serial form (J. Besag, "On the statistical analysis of dirty pictures", *JRSS B* 48, 1986, 259–302). The image, noise draw and evaluation design are course-constructed.